In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Lambda, RepeatVector, TimeDistributed
from sklearn.preprocessing import MinMaxScaler
import warnings

warnings.filterwarnings("ignore")

# --- 1. Configuration ---
INPUT_FILE = '../../data/data_with_features.csv'
OUTPUT_FILE = '../../data/syntheticData/synthetic_data_vae.csv'
TARGET_COLUMN = 'cumulative_mite_count'
GROUP_COLUMN = 'treatment_id'

# VAE & Model Parameters
LATENT_DIM = 16
LSTM_UNITS = 64
EPOCHS = 200
BATCH_SIZE = 32

# --- 2. Data Preparation (This part is unchanged) ---
print("--- ⚙️ Preparing Data for VAE ---")
df = pd.read_csv(INPUT_FILE)
grouped = df.groupby(GROUP_COLUMN)

original_series_list = [g[TARGET_COLUMN].values for _, g in grouped]
original_lengths = [len(s) for s in original_series_list]
max_len = max(original_lengths)
print(f"   Found {len(original_series_list)} time series groups.")
print(f"   Maximum series length is {max_len}. All series will be padded to this length.")

padded_series = np.array([np.pad(s, (0, max_len - len(s)), 'constant', constant_values=0) for s in original_series_list])

scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(padded_series.T).T
data_for_vae = scaled_data.reshape(len(scaled_data), max_len, 1)
print(f"   Data prepared with shape: {data_for_vae.shape}\n")


# --- 3. Build the VAE Model (Modern, Subclassing Approach) ---
print("--- 🏗️ Building VAE Model (Modern Keras API) ---")

# Define the sampling function (reparameterization trick)
def sampling(args):
    z_mean, z_log_var = args
    batch = tf.shape(z_mean)[0]
    dim = tf.shape(z_mean)[1]
    epsilon = tf.keras.backend.random_normal(shape=(batch, dim))
    return z_mean + tf.exp(0.5 * z_log_var) * epsilon

# Define the Encoder as a Keras Model
encoder_inputs = Input(shape=(max_len, 1), name='encoder_input')
x = LSTM(LSTM_UNITS, activation='relu')(encoder_inputs)
z_mean = Dense(LATENT_DIM, name='z_mean')(x)
z_log_var = Dense(LATENT_DIM, name='z_log_var')(x)
z = Lambda(sampling, name='z_sampler')([z_mean, z_log_var])
encoder = Model(encoder_inputs, [z_mean, z_log_var, z], name='encoder')

# Define the Decoder as a Keras Model
latent_inputs = Input(shape=(LATENT_DIM,), name='decoder_input')
x = RepeatVector(max_len)(latent_inputs)
x = LSTM(LSTM_UNITS, activation='relu', return_sequences=True)(x)
decoder_outputs = TimeDistributed(Dense(1))(x)
decoder = Model(latent_inputs, decoder_outputs, name='decoder')

# Define the VAE as a custom Model class to handle the loss
class VAE(Model):
    def __init__(self, encoder, decoder, **kwargs):
        super(VAE, self).__init__(**kwargs)
        self.encoder = encoder
        self.decoder = decoder
        self.total_loss_tracker = tf.keras.metrics.Mean(name="total_loss")
        self.reconstruction_loss_tracker = tf.keras.metrics.Mean(name="reconstruction_loss")
        self.kl_loss_tracker = tf.keras.metrics.Mean(name="kl_loss")

    @property
    def metrics(self):
        return [self.total_loss_tracker, self.reconstruction_loss_tracker, self.kl_loss_tracker]

    def train_step(self, data):
        with tf.GradientTape() as tape:
            z_mean, z_log_var, z = self.encoder(data)
            reconstruction = self.decoder(z)
            reconstruction_loss = tf.reduce_mean(
                tf.reduce_sum(tf.keras.losses.mse(data, reconstruction), axis=1)
            )
            kl_loss = -0.5 * (1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var))
            kl_loss = tf.reduce_mean(tf.reduce_sum(kl_loss, axis=1))
            total_loss = reconstruction_loss + kl_loss
        
        grads = tape.gradient(total_loss, self.trainable_weights)
        self.optimizer.apply_gradients(zip(grads, self.trainable_weights))
        
        self.total_loss_tracker.update_state(total_loss)
        self.reconstruction_loss_tracker.update_state(reconstruction_loss)
        self.kl_loss_tracker.update_state(kl_loss)
        
        return {
            "loss": self.total_loss_tracker.result(),
            "reconstruction_loss": self.reconstruction_loss_tracker.result(),
            "kl_loss": self.kl_loss_tracker.result(),
        }

# Instantiate and compile the VAE
vae = VAE(encoder, decoder)
vae.compile(optimizer=tf.keras.optimizers.Adam())
print("   Model built and compiled successfully.\n")


# --- 4. Train the VAE ---
print(f"--- 🏋️ Training VAE for {EPOCHS} epochs ---")
vae.fit(data_for_vae, epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=1)
print("   Training complete.\n")


# --- 5. Generate Synthetic Data ---
print("--- ✨ Generating New Synthetic Data ---")
num_samples_to_generate = len(original_series_list)
random_latent_vectors = np.random.normal(size=(num_samples_to_generate, LATENT_DIM))

# Use the decoder part of the trained VAE to generate data
generated_scaled = vae.decoder.predict(random_latent_vectors)

generated_padded = scaler.inverse_transform(generated_scaled.reshape(-1, max_len).T).T

generated_final = []
for i, length in enumerate(original_lengths):
    series = generated_padded[i, :length]
    series = np.maximum(0, series)
    series = np.maximum.accumulate(series)
    generated_final.append(series)
print("   Synthetic data generated and post-processed.\n")


# --- 6. Save the New Dataset ---
print(f"--- 💾 Saving to {OUTPUT_FILE} ---")
synthetic_df_list = []
original_groups = [g for _, g in grouped]

for i, group_df in enumerate(original_groups):
    temp_df = group_df.copy()
    temp_df[TARGET_COLUMN] = generated_final[i]
    synthetic_df_list.append(temp_df)

final_synthetic_df = pd.concat(synthetic_df_list, ignore_index=True)
final_synthetic_df.to_csv(OUTPUT_FILE, index=False)
print(f"   Successfully saved synthetic data. Final shape: {final_synthetic_df.shape}")
print("\n--- ✅ VAE Process Complete ---")

--- ⚙️ Preparing Data for VAE ---
   Found 224 time series groups.
   Maximum series length is 29. All series will be padded to this length.
   Data prepared with shape: (224, 29, 1)

--- 🏗️ Building VAE Model (Modern Keras API) ---

   Model built and compiled successfully.

--- 🏋️ Training VAE for 200 epochs ---
Epoch 1/200
7/7 ━━━━━━━━━━━━━━━━━━━━ 5s 27ms/step - kl_loss: 0.0033 - loss: 7.2637 - reconstruction_loss: 7.2604
Epoch 2/200
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - kl_loss: 0.0086 - loss: 6.3997 - reconstruction_loss: 6.3911
Epoch 3/200
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - kl_loss: 0.0159 - loss: 5.3381 - reconstruction_loss: 5.3222
Epoch 4/200
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - kl_loss: 0.0261 - loss: 5.1431 - reconstruction_loss: 5.1170
Epoch 5/200
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - kl_loss: 0.0312 - loss: 4.9159 - reconstruction_loss: 4.8847
Epoch 6/200
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - kl_loss: 0.0285 - loss: 4.8311 - reconstruction_loss: 4.8025
Epoch 

: 